# 第3章 前馈神经网络 · 参考答案



本册为依据课程材料新编的训练题，不是教师公布的考试原题。

## 材料依据
- [[课程/深度学习/PPT/Chapter3/Chapter3.md]]（神经元、激活、逻辑门、前向与反向传播、万能近似定理）
- [[课程/深度学习/PPT/Chapter3(1)/Chapter3(1).md]]（补充：向量输出 Jacobian 与标量损失梯度）
- [[课程/深度学习/课堂笔记/2026-09-14-第5节-神经网络反向传播算法推导与复杂度分析讲座.md]]（第三至九节）
- [[课程/深度学习/课堂笔记/2026-09-16-第6节-深度学习优化理论讲座.md]]（第二、三节：逼近存在性与训练的区别）
- [[课程/深度学习/作业/Homework02/Homework02.md]]（Exercise 2、3、7；保留任务，明确数组容器校注）
- [[课程/深度学习/作业/第1-6章-课件与课堂提问汇编作业.md]]（C3-01、03—11、A-04、05、15）
- [[课程/深度学习/演示代码/Chapter4(1).py]]（后续自动微分示范，并非独立的第3章演示）
- [[课程/深度学习/PPT/2026深度学习第一次上机考试规则与命令表v1.1 (1)/2026深度学习第一次上机考试规则与命令表v1.1 (1).md]]（30分钟、100分、无代码补全及API范围）


## 模拟要求

计时 **30 分钟，100 分**；一道完整编程题、多个小问，**从导入库开始独立编写，不做代码补全**。先作答，再运行自检。题面中的接口名称仅用于统一自检，不提供实现框架。

本地 PNG 写入当前目录 `output/`（请用 pathlib 创建目录）；正式考试若题目指定 `/output/Qxxx.png`，须严格使用正式路径和文件名。练习中的输出目录是本地适配，不是正式考试环境的改动。


## Q3B 双通道校准网络的标量损失与 PyTorch 梯度审计

**30分钟，100分；依据 Chapter3(1)向量输出与9月14日第九节新编；与A卷不同，此卷用自动微分审计双输出网络，不做训练。**

有6个三维输入 `X=[[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,-1,.5],[-.5,.5,1]]`，目标两通道为 `T[:,0]=X[:,0]+X[:,2]`、`T[:,1]=X[:,1]-X[:,2]`。
网络 `3→3→2`，隐藏tanh、输出恒等；固定
`W1=[[.5,0,-.5],[0,1,.25],[-.5,.5,1]]`、`b1=[.1,-.2,.3]`、
`W2=[[1,-.5,.25],[-.25,.75,1]]`、`b2=[.2,-.1]`。

1. **25分/约7分钟**：用 `nn.Module` 定义 `CalibrationNet`，含 `fc1,act,fc2`；float64、CPU，设置上述参数。创建 `model3b` 并打印参数数目20、批量输出形状及 `pred3b`。
2. **30分/约8分钟**：计算每个样本两通道平方误差之和 `sample_losses3b`（形状6），取均值 `loss3b`（不是把12个元素平均）。清梯度后只对标量loss一次反传；保存 `grad3b=model3b.fc1.weight.grad.clone()`，打印所有参数梯度。
3. **25分/约7分钟**：用 `torch.no_grad()` 下的中心差分对 `fc1.weight[0,2]` 校准，$h=10^{-6}$，恢复权重；保存 `fd3b`、`fd_error3b`，解释为何不必计算每个输出的完整Jacobian。
4. **20分/约8分钟**：用英文标签绘制两通道残差随样本编号的曲线，保存 `output/Q3B.png`，打印损失最大样本编号（Python零起始）。

允许命令表中的 `.double()`、`no_grad()`、`detach()`；测试没有前序环境单元，须自行导入。


In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from pathlib import Path
class CalibrationNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1=nn.Linear(3,3);self.act=nn.Tanh();self.fc2=nn.Linear(3,2)
    def forward(self,x):
        return self.fc2(self.act(self.fc1(x)))
model3b=CalibrationNet().double()
with torch.no_grad():
    model3b.fc1.weight[:]=torch.tensor([[.5,0,-.5],[0,1,.25],[-.5,.5,1]],dtype=torch.float64)
    model3b.fc1.bias[:]=torch.tensor([.1,-.2,.3],dtype=torch.float64)
    model3b.fc2.weight[:]=torch.tensor([[1,-.5,.25],[-.25,.75,1]],dtype=torch.float64)
    model3b.fc2.bias[:]=torch.tensor([.2,-.1],dtype=torch.float64)
X3b=torch.tensor([[0.,0.,0.],[1.,0.,0.],[0.,1.,0.],[0.,0.,1.],[1.,-1.,.5],[-.5,.5,1.]],dtype=torch.float64)
T3b=torch.stack((X3b[:,0]+X3b[:,2],X3b[:,1]-X3b[:,2]),dim=1)
pred3b=model3b(X3b)
sample_losses3b=((pred3b-T3b)**2).sum(dim=1)
loss3b=sample_losses3b.mean()
optimizer3b=torch.optim.SGD(model3b.parameters(),lr=.01)  # used only for zero_grad, no step
optimizer3b.zero_grad();loss3b.backward()
grad3b=model3b.fc1.weight.grad.clone()
print('parameters',sum(p.numel() for p in model3b.parameters()),'shape',pred3b.shape,'prediction',pred3b.detach())
print('sample losses',sample_losses3b.detach(),'loss',loss3b.item())
for p in model3b.parameters(): print('parameter gradient',p.grad)
h3b=1e-6
with torch.no_grad():
    saved=model3b.fc1.weight[0,2].item()
    model3b.fc1.weight[0,2]=saved+h3b
    fp=((model3b(X3b)-T3b)**2).sum(dim=1).mean().item()
    model3b.fc1.weight[0,2]=saved-h3b
    fm=((model3b(X3b)-T3b)**2).sum(dim=1).mean().item()
    model3b.fc1.weight[0,2]=saved
fd3b=(fp-fm)/(2*h3b)
fd_error3b=abs(fd3b-grad3b[0,2].item())
print('gradient / finite difference / error',grad3b[0,2].item(),fd3b,fd_error3b)
residual3b=(pred3b-T3b).detach().numpy()
plt.figure()
for j in range(2): plt.plot(np.arange(6),residual3b[:,j],label=f'channel {j}')
plt.xlabel('sample index');plt.ylabel('prediction - target');plt.legend()
Path('output').mkdir(exist_ok=True);plt.savefig('output/Q3B.png');plt.close()
print('worst sample index',sample_losses3b.argmax().item())


### 解析与点评

**评分点评**：25分看网络类、参数与批量形状；30分看按“先通道求和、后样本平均”的归约，误用整体MSE会使损失和梯度都少一半；25分看差分、权重恢复及标量损失解释；20分看残差与输出文件。`backward` 直接通过标量损失组合所有输出的链式贡献，等价于向量-Jacobian乘积，不需要显式保存每个样本每个通道的全部参数Jacobian。


### 自检（作答后运行）

这些是公开的代表性检查，不替代评分细则、图表检查与书面解释；通过不代表已满足全部要求。


In [ ]:
import numpy as np
import torch
from pathlib import Path
_check_X3b = torch.tensor([[0,0,0],[1,0,0],[0,1,0],[0,0,1],[1,-1,.5],[-.5,.5,1]], dtype=torch.float64)
_check_T3b = torch.stack((_check_X3b[:,0]+_check_X3b[:,2],_check_X3b[:,1]-_check_X3b[:,2]), dim=1)
assert sum(p.numel() for p in model3b.parameters())==20
assert pred3b.shape==(6,2) and sample_losses3b.shape==(6,) and fd_error3b<1e-7
with torch.no_grad():
    hh=torch.tanh(_check_X3b@model3b.fc1.weight.T+model3b.fc1.bias)
    rr=pred3b-_check_T3b
    pp=(2/6)*(rr@model3b.fc2.weight)*(1-hh**2)
    manual=pp.T@_check_X3b
assert np.max(np.abs(manual.numpy()-grad3b.numpy()))<1e-12
expected_first=np.tanh(np.array([.1,-.2,.3]))@np.array([[1.,-.25],[-.5,.75],[.25,1.]])+np.array([.2,-.1])
assert np.allclose(pred3b.detach().numpy()[0],expected_first,atol=1e-12)
assert abs(loss3b.item()-sample_losses3b.detach().mean().item())<1e-12
assert abs(model3b.fc1.weight[0,2].item()+.5)<1e-12
assert Path('output/Q3B.png').stat().st_size>1000

print("自检通过")
